# Homework 3 Part 1

## Setup

### Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit, broadcast, hash, max, min, avg, count, first

### Create Spark Session

In [ ]:
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

In [ ]:
spark

### Disable Automatic Broadcast Join

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

### Update system configurations to enable bucket join and preserve data grouping

In [ ]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') 
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

## Build Spark Job

### Delete DDLs for each table for a fresh start

In [ ]:
%%sql

DROP TABLE bootcamp.matches;

In [ ]:
%%sql

DROP TABLE bootcamp.match_details;

In [ ]:
%%sql

DROP TABLE bootcamp.medals_matches_players;

### First establish DDLs for each table.

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.medals_matches_players (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.match_details (
    match_id STRING, 
    player_gamertag STRING, 
    previous_spartan_rank STRING, 
    spartan_rank STRING, 
    previous_total_xp STRING, 
    total_xp STRING, 
    previous_csr_tier STRING, 
    previous_csr_designation STRING, 
    previous_csr STRING, 
    previous_csr_percent_to_next_tier STRING, 
    previous_csr_rank STRING, 
    current_csr_tier STRING, 
    current_csr_designation STRING, 
    current_csr STRING, 
    current_csr_percent_to_next_tier STRING, 
    current_csr_rank STRING, 
    player_rank_on_team STRING, 
    player_finished STRING, 
    player_average_life STRING, 
    player_total_kills INTEGER, 
    player_total_headshots STRING, 
    player_total_weapon_damage STRING, 
    player_total_shots_landed STRING, 
    player_total_melee_kills STRING, 
    player_total_melee_damage STRING, 
    player_total_assassinations STRING, 
    player_total_ground_pound_kills STRING, 
    player_total_shoulder_bash_kills STRING, 
    player_total_grenade_damage STRING, 
    player_total_power_weapon_damage STRING, 
    player_total_power_weapon_grabs STRING, 
    player_total_deaths STRING,
    player_total_assists STRING, 
    player_total_grenade_kills STRING, 
    did_win STRING, 
    team_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.matches (
    match_id STRING, 
    mapid STRING, 
    is_team_game STRING, 
    playlist_id STRING, 
    game_variant_id STRING, 
    is_match_over STRING, 
    completion_date STRING, 
    match_duration STRING, 
    game_mode STRING, 
    map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

### Write CSV data to tables bucketing on match_id

In [ ]:
matches_df = spark.read.option("header", "true").csv("/home/iceberg/data/matches.csv")
matches_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.matches")

In [ ]:
match_details_df = spark.read.option("header", "true").csv("/home/iceberg/data/match_details.csv").withColumnRenamed("player_gamertag", "match_details_player_gamertag")
match_details_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.match_details")

In [ ]:
medals_matches_players_df = spark.read.option("header", "true").csv("/home/iceberg/data/medals_matches_players.csv").withColumnRenamed("player_gamertag", "medals_matches_players_player_gamertag")
medals_matches_players_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.medals_matches_players")

### Read Data From Tables

In [ ]:
matches_df = spark.table("bootcamp.matches")
matches_df

In [ ]:
match_details_df = spark.table("bootcamp.match_details")
match_details_df

In [ ]:
medal_matches_players_df = spark.table("bootcamp.medals_matches_players")
medal_matches_players_df

### Bucket Join Medals Matches Players, Match Details, and Matches on 16 buckets

In [ ]:
joined_df = medal_matches_players_df.join(match_details_df, on="match_id", how="inner")

joined_df = joined_df.join(matches_df, on="match_id")

joined_df.explain()

### Read medals and maps tables from CSV for broadcasting

In [ ]:
medals_df = spark.read.option("header", "true").csv("/home/iceberg/data/medals.csv").withColumnRenamed("name", "medal_name").withColumnRenamed("description", "medal_description")
medals_df

In [ ]:
maps_df = spark.read.option("header", "true").csv("/home/iceberg/data/maps.csv").withColumnRenamed("name", "map_name").withColumnRenamed("description", "map_description")
maps_df

### Explicitly broadcast JOINs medals and maps

In [ ]:
joined_df = joined_df.join(broadcast(medals_df), on="medal_id", how="inner")

joined_df = joined_df.join(broadcast(maps_df), on="mapid", how="inner")

joined_df.explain()

## Aggregate the joined data frame to figure out questions like:

### Which player averages the most kills per game?

In [ ]:
most_kills_per_game_df = joined_df \
    .groupBy("match_details.match_details_player_gamertag") \
    .agg(avg(col("match_details.player_total_kills")).alias("avg_kills_per_game")) \
    .orderBy(col("avg_kills_per_game").desc())
most_kills_per_game_df.show()

### Which playlist gets played the most?

In [ ]:
most_played_playlist_df = joined_df \
.groupBy("matches.playlist_id") \
.agg(count("matches.playlist_id").alias("played_count")) \
.orderBy(col("played_count").desc())
most_played_playlist_df.show()

### Which map gets played the most?

In [ ]:
most_played_maps_df = joined_df \
    .groupBy("mapid") \
    .agg(count("*").alias("played_count"), first("map_name").alias("map_name")) \
    .orderBy(col("played_count").desc())
most_played_maps_df.show()

### Which map do players get the most Killing Spree medals on?

In [ ]:
most_kill_spree_per_map_maps_df = joined_df \
    .filter(col("medal_name") == "Killing Spree") \
    .groupBy("mapid", "medal_id") \
    .agg(avg("count").alias("avg_killing_spree"), first("map_name").alias("map_name")) \
    .orderBy(col("avg_killing_spree").desc())

most_kill_spree_per_map_maps_df.show()

## With the aggregated data set
Try different .sortWithinPartitions to see which has the smallest data size (hint: playlists and maps are both very low cardinality)

In [ ]:
%%sql

DROP TABLE bootcamp.aggregated_data;

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.aggregated_data (
    mapid STRING, 
    medal_id STRING, 
    match_id STRING, 
    medals_matches_players_player_gamertag STRING, 
    count STRING, 
    match_details_player_gamertag STRING, 
    previous_spartan_rank STRING, 
    spartan_rank STRING, 
    previous_total_xp STRING, 
    total_xp STRING, 
    previous_csr_tier STRING, 
    previous_csr_designation STRING, 
    previous_csr STRING, 
    previous_csr_percent_to_next_tier STRING, 
    previous_csr_rank STRING, 
    current_csr_tier STRING, 
    current_csr_designation STRING, 
    current_csr STRING, 
    current_csr_percent_to_next_tier STRING, 
    current_csr_rank STRING, 
    player_rank_on_team STRING, 
    player_finished STRING, 
    player_average_life STRING, 
    player_total_kills STRING, 
    player_total_headshots STRING, 
    player_total_weapon_damage STRING, 
    player_total_shots_landed STRING, 
    player_total_melee_kills STRING, 
    player_total_melee_damage STRING, 
    player_total_assassinations STRING, 
    player_total_ground_pound_kills STRING, 
    player_total_shoulder_bash_kills STRING, 
    player_total_grenade_damage STRING, 
    player_total_power_weapon_damage STRING, 
    player_total_power_weapon_grabs STRING, 
    player_total_deaths STRING, 
    player_total_assists STRING, 
    player_total_grenade_kills STRING, 
    did_win STRING, 
    team_id STRING, 
    is_team_game STRING, 
    playlist_id STRING, 
    game_variant_id STRING, 
    is_match_over STRING, 
    completion_date STRING, 
    match_duration STRING, 
    game_mode STRING, 
    map_variant_id STRING, 
    sprite_uri STRING, 
    sprite_left STRING, 
    sprite_top STRING, 
    sprite_sheet_width STRING, 
    sprite_sheet_height STRING, 
    sprite_width STRING, 
    sprite_height STRING, 
    classification STRING, 
    medal_description STRING, 
    medal_name STRING, 
    difficulty STRING, 
    map_name STRING, 
    map_description STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

In [ ]:
joined_df.write \
    .format("iceberg") \
    .bucketBy(16, "match_id") \
    .mode("overwrite") \
    .saveAsTable("bootcamp.aggregated_data")

In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM bootcamp.aggregated_data.files;

### Parition on matches.playlist_id (low cardinality)

In [ ]:
joined_df.sortWithinPartitions("matches.playlist_id") \
    .writeTo("bootcamp.aggregated_data") \
    .using("iceberg") \
    .option("overwrite-mode", "dynamic") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM bootcamp.aggregated_data.files;

### Parition on maps.map_id (low cardinality)

In [ ]:
joined_df.sortWithinPartitions("matches.mapid") \
    .writeTo("bootcamp.aggregated_data") \
    .using("iceberg") \
    .option("overwrite-mode", "dynamic") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM bootcamp.aggregated_data.files;

### Parition on matches.playlist_id and maps.map_id (low cardinality)

In [ ]:
joined_df.sortWithinPartitions("matches.playlist_id", "matches.mapid") \
    .writeTo("bootcamp.aggregated_data") \
    .using("iceberg") \
    .option("overwrite-mode", "dynamic") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM bootcamp.aggregated_data.files;

### Parition on matches.match_id (high cardinality)

In [ ]:
joined_df.sortWithinPartitions("matches.match_id") \
    .writeTo("bootcamp.aggregated_data") \
    .using("iceberg") \
    .option("overwrite-mode", "dynamic") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM bootcamp.aggregated_data.files;